In [ ]:
!date

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from pybedtools import BedTool, Interval

import subprocess

import scanpy as sc
import anndata

from concurrent.futures import ProcessPoolExecutor, as_completed
from tqdm import tqdm

from scipy.stats import zscore
import pyranges as pr

In [ ]:
sc.set_figure_params(figsize=(5,5), frameon=False)
sc.settings.verbosity = 3

In [ ]:
projdir = '/u/project/cluo/terencew/igvf/2023_YR2/snmCT/pseudobulk_allc'
donors = list(np.loadtxt(f'{projdir}/txt/donors.txt', dtype=str))
tmp_order = ['Start', 'Sendai', 'Delayed', 'Fail1', 'Inter1', 'Inter2', 'Fail2', 'IPS']
clusts = [f'{x}_{y}' for x,y in zip(np.repeat(tmp_order, len(donors)), np.tile(donors, len(tmp_order)))]

In [ ]:
indir = f'{projdir}/pmds/dnmtools/rna_cluster_donor'
pmds = pd.read_csv(f'{indir}/merged/start_merged.filt_mcg.sorted.bed.gz', sep='\t', header=None)
pmds.index = [f'{x}_{y}' for x,y in zip(pmds[0], pmds[1])]
pmds.columns = ['chrom', 'start', 'end']
pmds.shape

### load in DEGs

In [ ]:
degs_path = '/u/project/cluo/terencew/igvf/2023_YR2/multiome/csv/degs/justin_degs_v7.csv'
degs = pd.read_csv(degs_path, sep='\t')
degs = degs[degs['p_val_adj'] < 0.05]
degs.shape

In [ ]:
degs.head()

### rna expression lower on average in PMDs?

In [ ]:
indir = '/u/project/cluo/terencew/igvf/2023_YR2/multiome'
adata = sc.read_h5ad(f'{indir}/h5ad/rna/raw/intersect_atac.h5ad')

clusters = pd.read_csv(f'{indir}/csv/clusters/rna_leiden_v7.csv', sep='\t', index_col=0)
adata.obs['coarse'] = clusters['coarse_v7'].reindex(adata.obs.index)
adata.obs['coarse_donor'] = [f'{x}_{y}' for x,y in zip(adata.obs['coarse'], adata.obs['assignment'])]

In [ ]:
adata

In [ ]:
tmp_cols = ['chrom', 'start', 'end', 'gene']
genes_bed_path = '/u/project/cluo/terencew/reference/hg38_igvf/bed/genes/gencode.v43.autosome.dedup.names.bed.gz'
genes_bed = pd.read_csv(genes_bed_path, sep='\t', header=None)
genes_bed.index = genes_bed[3]
genes_bed.columns = tmp_cols
genes_bed.shape

In [ ]:
genes_bed.head()

In [ ]:
### subset adata to autosome
adata.var['chrom'] = genes_bed.reindex(adata.var.index)['chrom']
autosome = [f'chr{x}' for x in range(1,23)]
mask = adata.var['chrom'].isin(autosome)
auto_adata = adata[:,mask]
auto_adata

In [ ]:
def process_pseudo(s):
    tmp_adata = auto_adata[auto_adata.obs['coarse_donor'] == s]
    mtx = tmp_adata.X.sum(axis=0)
    return mtx

with ProcessPoolExecutor() as executor:
    results = list(tqdm(executor.map(process_pseudo, clusts), total=len(clusts)))
    
pseudo_merged = np.vstack(results)
pseudo_df = pd.DataFrame(pseudo_merged, index=clusts, columns=auto_adata.var.index)
pseudo_df.shape

In [ ]:
denom = pseudo_df.sum(axis=1)
norm_pseudo = pseudo_df.div(denom, axis=0) * 1e6
norm_pseudo.shape

In [ ]:
log_pseudo = np.log(norm_pseudo + 1)
log_pseudo.shape

### split all genes into PMDs/non-PMDS

In [ ]:
pmds_bt = BedTool.from_dataframe(pmds)
genes_bt = BedTool.from_dataframe(genes_bed)
boundaries = []
for p in pmds_bt:
    boundaries.append(Interval(p.chrom, p.start, p.start+1))
    boundaries.append(Interval(p.chrom, p.end-1, p.end))

boundaries = BedTool(boundaries)
res = genes_bt.closest(boundaries, d=True)
genes_dist = res.to_dataframe()
genes_dist.index = genes_dist['name']
genes_dist = genes_dist[~genes_dist.index.duplicated()]
genes_dist.shape

In [ ]:
genes_dist.head()

In [ ]:
pmds_bt = BedTool.from_dataframe(pmds)
genes_bt = BedTool.from_dataframe(genes_bed)
genes_pmds = genes_bt.intersect(pmds_bt, wa=True).to_dataframe()
genes_pmds.index = genes_pmds['name']
genes_pmds.shape

In [ ]:
s = 'Start'
mask = [x.split('_')[0] == s for x in norm_pseudo.index]

tmp_pseudo = norm_pseudo[mask]
tmp_pseudo.shape

In [ ]:
merged_expr = pd.DataFrame(np.log(tmp_pseudo.mean(axis=0) + 1))
merged_expr.shape

In [ ]:
mask = merged_expr.index.isin(genes_pmds['name'])
merged_pmds = merged_expr[mask].copy()
merged_pmds.shape

In [ ]:
mask = ~merged_expr.index.isin(genes_pmds['name'])
merged_comp = merged_expr[mask].copy()
merged_comp.shape

In [ ]:
merged_pmds['type'] = 'pmds'
merged_comp['type'] = 'non-pmds'
to_plot = pd.concat([merged_pmds, merged_comp])
to_plot.columns = ['expr', 'type']
mask = to_plot['expr'] != 0
to_plot = to_plot[mask]

###
to_plot['dist'] = genes_dist.reindex(to_plot.index)['blockSizes']
mask = to_plot['type'] == 'pmds'
to_plot.loc[mask,'dist'] = -1 * to_plot.loc[mask,'dist']

to_plot.shape

In [ ]:
to_plot.head()

In [ ]:
fig, axes = plt.subplots(1, figsize=(8, 4))

x_lab = 'dist'
y_lab = 'expr'
hue = 'type'

ax = sns.histplot(data=to_plot, x=x_lab, y=y_lab)
ax.set_xlabel('', fontsize=16)
ax.set_ylabel('Expression (log CPM)', fontsize=16)
ax.set_title('Start - all gene expression inside/outside of PMDs', fontsize=14)

ax.tick_params(axis='x', labelsize=14, labelrotation=0)

In [ ]:
fig, axes = plt.subplots(1, figsize=(8, 4))

x_lab = 'dist'
y_lab = 'expr'
hue = 'type'

ax = sns.scatterplot(data=to_plot, x=x_lab, y=y_lab, size=1)
ax.set_xlabel('', fontsize=16)
ax.set_ylabel('Expression (log CPM)', fontsize=16)
ax.set_title('Start - all gene expression inside/outside of PMDs', fontsize=14)

ax.tick_params(axis='x', labelsize=14, labelrotation=0)

In [ ]:
outdir = '/u/project/cluo/terencew/igvf/2023_YR2/final_figures/csv'
to_plot.to_csv(f'{outdir}/figure2/rna_expr_genes_dist_start.csv', sep='\t', index=None)

### try DEGs now

In [ ]:
s = 'Start'
tmp_degs = degs[degs['coarse_v7'] == s]
tmp_genes = genes_bed.reindex(tmp_degs['gene'])
tmp_genes = tmp_genes[~tmp_genes['chrom'].isna()]
tmp_genes['start'] = tmp_genes['start'].astype(int)
tmp_genes['end'] = tmp_genes['end'].astype(int)
tmp_genes.shape

In [ ]:
tmp_genes.head()

In [ ]:
pmds_bt = BedTool.from_dataframe(pmds)
genes_bt = BedTool.from_dataframe(tmp_genes)
genes_pmds = genes_bt.intersect(pmds_bt, wa=True).to_dataframe()
genes_pmds.index = genes_pmds['name']

genes_comp = genes_bt.intersect(pmds_bt, wa=True, v=True).to_dataframe()
genes_comp.index = genes_comp['name']

mask = [x.split('_')[0] == s for x in norm_pseudo.index]
tmp_pseudo = norm_pseudo[mask]
tmp_pseudo.shape

merged_expr = pd.DataFrame(np.log(tmp_pseudo.mean(axis=0) + 1))
mask = merged_expr.index.isin(genes_pmds['name'])
merged_pmds = merged_expr[mask].copy()

mask = merged_expr.index.isin(genes_comp['name'])
merged_comp = merged_expr[mask].copy()
merged_pmds.shape, merged_comp.shape

In [ ]:
merged_pmds['type'] = 'pmds'
merged_comp['type'] = 'non-pmds'
to_plot = pd.concat([merged_pmds, merged_comp])
to_plot.columns = ['expr', 'type']
mask = to_plot['expr'] != 0
to_plot = to_plot[mask]

###
to_plot['dist'] = genes_dist.reindex(to_plot.index)['blockSizes']
mask = to_plot['type'] == 'pmds'
to_plot.loc[mask,'dist'] = -1 * to_plot.loc[mask,'dist']

to_plot.shape

In [ ]:
tmp = to_plot[to_plot['type'] == 'pmds']
tmp['dist'], tmp['dist'].value_counts()

In [ ]:
fig, axes = plt.subplots(1, figsize=(8, 4))

x_lab = 'dist'
y_lab = 'expr'
hue = 'type'

ax = sns.scatterplot(data=to_plot, x=x_lab, y=y_lab, size=1)
ax.set_xlabel('', fontsize=16)
ax.set_ylabel('Expression (log CPM)', fontsize=16)
ax.set_title('Start - DEG gene expression inside/outside of PMDs', fontsize=14)

ax.tick_params(axis='x', labelsize=14, labelrotation=0)

### do it for all clusters

In [ ]:
clust_list = []
for s in tmp_order:
    tmp_degs = degs[degs['coarse_v7'] == s]
    tmp_genes = genes_bed.reindex(tmp_degs['gene'])
    tmp_genes = tmp_genes[~tmp_genes['chrom'].isna()]
    tmp_genes['start'] = tmp_genes['start'].astype(int)
    tmp_genes['end'] = tmp_genes['end'].astype(int)

    pmds_bt = BedTool.from_dataframe(pmds)
    genes_bt = BedTool.from_dataframe(tmp_genes)
    genes_pmds = genes_bt.intersect(pmds_bt, wa=True).to_dataframe()
    genes_pmds.index = genes_pmds['name']

    genes_comp = genes_bt.intersect(pmds_bt, wa=True, v=True).to_dataframe()
    genes_comp.index = genes_comp['name']

    mask = [x.split('_')[0] == s for x in norm_pseudo.index]
    tmp_pseudo = norm_pseudo[mask]
    tmp_pseudo.shape

    merged_expr = pd.DataFrame(np.log(tmp_pseudo.mean(axis=0) + 1))
    mask = merged_expr.index.isin(genes_pmds['name'])
    merged_pmds = merged_expr[mask].copy()

    mask = merged_expr.index.isin(genes_comp['name'])
    merged_comp = merged_expr[mask].copy()
    merged_pmds.shape, merged_comp.shape
    merged_pmds['type'] = 'pmds'
    merged_comp['type'] = 'non-pmds'
    to_plot = pd.concat([merged_pmds, merged_comp])
    to_plot.columns = ['expr', 'type']

    ###
    to_plot['dist'] = genes_dist.reindex(to_plot.index)['blockSizes']
    mask = to_plot['type'] == 'pmds'
    to_plot.loc[mask,'dist'] = -1 * to_plot.loc[mask,'dist']

    to_plot['cluster'] = s
    clust_list.append(to_plot)
    
pmds_plot = pd.concat(clust_list)
pmds_plot.shape

In [ ]:
to_plot['type'].value_counts()

In [ ]:
fig, axes = plt.subplots(1, figsize=(8, 4))

x_lab = 'dist'
y_lab = 'expr'
hue = 'cluster'

ax = sns.scatterplot(data=pmds_plot, x=x_lab, y=y_lab, hue=hue, size=1)
ax.set_xlabel('', fontsize=16)
ax.set_ylabel('Expression (log CPM)', fontsize=16)
ax.set_title('Start - all gene expression inside/outside of PMDs', fontsize=14)

ax.tick_params(axis='x', labelsize=14, labelrotation=0)
sns.move_legend(ax, "upper left", bbox_to_anchor=(1, 1))

In [ ]:
outdir = '/u/project/cluo/terencew/igvf/2023_YR2/final_figures/csv'
pmds_plot.to_csv(f'{outdir}/figure3/rna_expr_degs_dist.csv', sep='\t', index=None)

### DEGs

In [ ]:
s = 'Start'
tmp_degs = degs[degs['coarse_v7'] == s]
tmp_genes = genes_bed.reindex(tmp_degs['gene'])
tmp_genes = tmp_genes[~tmp_genes['chrom'].isna()]
tmp_genes['start'] = tmp_genes['start'].astype(int)
tmp_genes['end'] = tmp_genes['end'].astype(int)
tmp_genes.shape

In [ ]:
pmds_bt = BedTool.from_dataframe(pmds)
genes_bt = BedTool.from_dataframe(tmp_genes)
genes_pmds = genes_bt.intersect(pmds_bt, wa=True).to_dataframe()
genes_pmds.index = genes_pmds['name']

genes_comp = genes_bt.intersect(pmds_bt, wa=True, v=True).to_dataframe()
genes_comp.index = genes_comp['name']
genes_pmds.shape, genes_comp.shape

In [ ]:
mask = [x.split('_')[0] == s for x in norm_pseudo.index]

tmp_pseudo = norm_pseudo[mask]
merged_expr = pd.DataFrame(np.log(tmp_pseudo.mean(axis=0) + 1))
mask = merged_expr.index.isin(genes_pmds['name'])
merged_pmds = merged_expr[mask].copy()
mask = merged_expr.index.isin(genes_comp['name'])
merged_comp = merged_expr[mask].copy()
merged_pmds.shape, merged_comp.shape

In [ ]:
merged_pmds['type'] = 'pmds'
merged_comp['type'] = 'non-pmds'
to_plot = pd.concat([merged_pmds, merged_comp])
to_plot.columns = ['expr', 'type']
to_plot.shape

In [ ]:
fig, axes = plt.subplots(1, figsize=(8, 4))

x_lab = 'type'
y_lab = 'expr'
hue = 'type'

ax = sns.boxplot(data=to_plot, x=x_lab, y=y_lab, showfliers=False)
ax.set_xlabel('', fontsize=16)
ax.set_ylabel('Expression (log CPM)', fontsize=16)
ax.set_title('Start - DEG expression inside/outside of PMDs', fontsize=14)

ax.tick_params(axis='x', labelsize=14, labelrotation=0)

### do this for all clusters

In [ ]:
clust_list = []
for s in tmp_order:
    ###
    tmp_degs = degs[degs['coarse_v7'] == s]
    tmp_genes = genes_bed.reindex(tmp_degs['gene'])
    tmp_genes = tmp_genes[~tmp_genes['chrom'].isna()]
    tmp_genes['start'] = tmp_genes['start'].astype(int)
    tmp_genes['end'] = tmp_genes['end'].astype(int)

    pmds_bt = BedTool.from_dataframe(pmds)
    genes_bt = BedTool.from_dataframe(tmp_genes)
    genes_pmds = genes_bt.intersect(pmds_bt, wa=True).to_dataframe()
    genes_pmds.index = genes_pmds['name']

    genes_comp = genes_bt.intersect(pmds_bt, wa=True, v=True).to_dataframe()
    genes_comp.index = genes_comp['name']
    
    ###
    mask = [x.split('_')[0] == s for x in norm_pseudo.index]
    tmp_pseudo = norm_pseudo[mask]

    ####
    merged_expr = pd.DataFrame(np.log(tmp_pseudo.mean(axis=0) + 1))
    mask = merged_expr.index.isin(genes_pmds['name'])
    merged_pmds = merged_expr[mask].copy()
    mask = merged_expr.index.isin(genes_comp['name'])
    merged_comp = merged_expr[mask].copy()
    
    ###
    merged_pmds['type'] = 'pmds'
    merged_comp['type'] = 'non-pmds'
    to_plot = pd.concat([merged_pmds, merged_comp])
    to_plot.columns = ['expr', 'type']
    mask = to_plot['expr'] != 0
    to_plot = to_plot[mask]
    to_plot['cluster'] = s
    clust_list.append(to_plot)
    
pmds_plot = pd.concat(clust_list)
pmds_plot.shape

In [ ]:
### interesting
fig, axes = plt.subplots(1, figsize=(8, 4))

x_lab = 'cluster'
y_lab = 'expr'
hue = 'type'

ax = sns.boxplot(data=pmds_plot, x=x_lab, y=y_lab, hue=hue, showfliers=False)
ax.set_xlabel('', fontsize=16)
ax.set_ylabel('Expression (log CPM)', fontsize=16)
ax.set_title('DEG expression inside/outside of PMDs', fontsize=14)

ax.tick_params(axis='x', labelsize=14, labelrotation=0)
sns.move_legend(ax, "upper left", bbox_to_anchor=(1, 1))

In [ ]:
outdir = '/u/project/cluo/terencew/igvf/2023_YR2/final_figures/csv'
pmds_plot.to_csv(f'{outdir}/figure3/rna_expr_degs_in_pmds.csv', sep='\t', index=None)

### are PMDs variably repressive between individuals?

In [ ]:
s = 'Start'
tmp_degs = degs[degs['coarse_v7'] == s]
tmp_genes = genes_bed.reindex(tmp_degs['gene'])
tmp_genes = tmp_genes[~tmp_genes['chrom'].isna()]
tmp_genes['start'] = tmp_genes['start'].astype(int)
tmp_genes['end'] = tmp_genes['end'].astype(int)

pmds_bt = BedTool.from_dataframe(pmds)
genes_bt = BedTool.from_dataframe(tmp_genes)
genes_pmds = genes_bt.intersect(pmds_bt, wa=True).to_dataframe()
genes_pmds.index = genes_pmds['name']

genes_comp = genes_bt.intersect(pmds_bt, wa=True, v=True).to_dataframe()
genes_comp.index = genes_comp['name']

In [ ]:
mask = [x.split('_')[0] == s for x in norm_pseudo.index]
clust_pseudo = log_pseudo[mask].T
clust_pseudo.shape

In [ ]:
mask = clust_pseudo.index.isin(genes_pmds['name'])
merged_pmds = clust_pseudo[mask].copy()
merged_pmds.shape

In [ ]:
mask = clust_pseudo.index.isin(genes_comp['name'])
merged_comp = clust_pseudo[mask].copy()
merged_comp.shape

In [ ]:
merged_pmds['type'] = 'pmds'
merged_comp['type'] = 'non-pmds'
concat = pd.concat([merged_pmds, merged_comp])
to_plot = concat.melt(id_vars=concat.columns[-1],value_vars=concat.columns[:-1])
to_plot['cluster'] = [x.split('_')[0] for x in to_plot['variable']]
to_plot['donor'] = [x.split('_')[1] for x in to_plot['variable']]
mask = to_plot['value'] != 0
to_plot = to_plot[mask]
to_plot.shape

In [ ]:
### interesting
fig, axes = plt.subplots(1, figsize=(8, 4))

x_lab = 'donor'
y_lab = 'value'
hue = 'type'

ax = sns.boxplot(data=to_plot, x=x_lab, y=y_lab, hue=hue,showfliers=False)
ax.set_xlabel('', fontsize=16)
ax.set_ylabel('Expression (log CPM)', fontsize=16)
ax.set_title('Start - DEG Gene expression inside/outside of PMDs', fontsize=14)

ax.tick_params(axis='x', labelsize=14, labelrotation=0)
sns.move_legend(ax, "upper left", bbox_to_anchor=(1, 1))

In [ ]:
clust_list = []
for s in tmp_order:
    ###
    tmp_degs = degs[degs['coarse_v7'] == s]
    tmp_genes = genes_bed.reindex(tmp_degs['gene'])
    tmp_genes = tmp_genes[~tmp_genes['chrom'].isna()]
    tmp_genes['start'] = tmp_genes['start'].astype(int)
    tmp_genes['end'] = tmp_genes['end'].astype(int)

    pmds_bt = BedTool.from_dataframe(pmds)
    genes_bt = BedTool.from_dataframe(tmp_genes)
    genes_pmds = genes_bt.intersect(pmds_bt, wa=True).to_dataframe()
    genes_pmds.index = genes_pmds['name']

    genes_comp = genes_bt.intersect(pmds_bt, wa=True, v=True).to_dataframe()
    genes_comp.index = genes_comp['name']
    
    ###
    mask = [x.split('_')[0] == s for x in norm_pseudo.index]
    clust_pseudo = log_pseudo[mask].T

    ###
    mask = clust_pseudo.index.isin(genes_pmds['name'])
    merged_pmds = clust_pseudo[mask].copy()
    mask = clust_pseudo.index.isin(genes_comp['name'])
    merged_comp = clust_pseudo[mask].copy()
    
    ###
    merged_pmds['type'] = 'pmds'
    merged_comp['type'] = 'non-pmds'
    concat = pd.concat([merged_pmds, merged_comp])
    to_plot = concat.melt(id_vars=concat.columns[-1],value_vars=concat.columns[:-1])
    to_plot['cluster'] = [x.split('_')[0] for x in to_plot['variable']]
    to_plot['donor'] = [x.split('_')[1] for x in to_plot['variable']]
    mask = to_plot['value'] != 0
    to_plot = to_plot[mask]
    to_plot.shape

    clust_list.append(to_plot)
    
pmds_plot = pd.concat(clust_list)
pmds_plot.shape

In [ ]:
pmds_plot.head()

In [ ]:
fig, axes = plt.subplots(4, 2, figsize=(16, 8))

x_lab = 'donor'
y_lab = 'value'
hue = 'type'

for i,ax in enumerate(axes.flatten()):
    s = tmp_order[i]
    to_plot = pmds_plot[pmds_plot['cluster'] == s]
    sns.boxplot(data=to_plot, x=x_lab, y=y_lab, hue=hue,showfliers=False, ax=ax)
    ax.set_xlabel('', fontsize=10)
    ax.set_ylabel('log CPM', fontsize=10)
    ax.set_title(f'{s}', fontsize=14)
    ax.legend_.remove()
    ax.tick_params(axis='x', labelsize=12, labelrotation=0)
    
plt.subplots_adjust(hspace=0.6, wspace=0.2)

In [ ]:
!date